https://docs.python.org/zh-cn/3.10/library/functools.html?highlight=partial#

https://www.cnblogs.com/tian777/p/17773050.html

functools 是 Python 标准库中的一个模块，提供了一系列用于操作函数和可调用对象的工具。它可以帮助你编写更简洁、更高效的代码，尤其是在处理函数的组合、装饰器、偏函数等方面。下面是一些 functools 模块中常用的功能和示例

### functools.partial 用于创建一个偏函数，
- 偏函数，把函数部分的参数固定下来，相当于为部分的参数添加了一个固定的默认值，形成一个新的函数并返回
- 从partial生成的新函数, 并且可以像原始对象一样对待，这样可以简化函数调用。

In [1]:
import functools

def add(a, b):
    return a + b

# 创建一个偏函数，固定第一个参数为5
add_five = functools.partial(add, 5)

print(add_five(10))  # 输出 15

15


### “给 json.dumps 打了个补丁，让它遇到 datetime / date / bytes 时不会炸，而是自动转成字符串。”
- json.dumps 把 Python 对象序列化成 JSON 字符串
- default=None：遇到不可序列化类型时调用的回调，上文自定义json_serial_fallback 就是塞这里。
- bytes、datetime、Decimal、UUID 等默认不支持

In [7]:
import base64,datetime, json, functools

def json_serial_fallback(obj):
    if isinstance(obj, (datetime.datetime, datetime.date)):
        return str(obj)
    if isinstance(obj, bytes):
         # 变成 ASCII 字符串，反序列化时再 b64decode 即可
        return {'__bytes__': base64.b64encode(obj).decode('ascii')}
    raise TypeError(f"{obj!r} is not JSON serializable")

json_dumps = functools.partial(json.dumps, default=json_serial_fallback)

data = {
    'time': datetime.datetime.now(),
    'date': datetime.date.today(),
    'blob': b'binary\xfc\x00\xff'
}

print(json_dumps(data, ensure_ascii=False, indent=2))  

{
  "time": "2026-01-15 13:07:28.384923",
  "date": "2026-01-15",
  "blob": {
    "__bytes__": "YmluYXJ5/AD/"
  }
}


## 延迟执行

In [9]:
import time
import functools

class DelayFunc:
    def __init__(self,duration,func):
        self.duration = duration
        self.func = func
    def __call__(self,*args,**kwargs):
        print(f"Wait for {self.duration} seconds...")
        time.sleep(self.duration)
        return self.func(*args,**kwargs)


def delay(duration):
    return functools.partial(DelayFunc,duration)


@delay(duration=2)
def add(a,b):
    return a + b

print(add(3,5))

Wait for 2 seconds...
8


### 回调函数添加参数
-别在交互式解释器里直接跑 multcessing.Pool 的任务函数；写成脚本文件再 python file.py 就正常了

In [ ]:
import logging
from multiprocessing import Pool
from functools import partial

def output_result(result,log=None):
    # 回调函数：当子进程计算完成后，主进程会拿到 result 并把它传进来；
    # 如果同时给了 log 就写一条 DEBUG 日志。
    if log is not None:
        # %r 用 repr() 打印，方便调试时看到值的类型细节
        log.debug('Got:%r',result)
def add(x,y):
    return x+y

if __name__ == '__main__':
    logging.basicConfig(level=logging.DEBUG)
    log = logging.getLogger('test')
    # 创建默认大小（CPU 核数）的进程池。
    p = Pool()
    '''
    1-把 add(3, 4) 这一任务异步扔进子进程。
    2-子进程算完后把返回值 7 交给主进程的回调函数。
    3-partial(output_result, log=log) 提前把 log 参数绑好，等回调时只接收一个 result
    '''
    p.apply_async(add,(3,4),callback=partial(output_result,log=log))
    p.close() # 关闭 Pool，不再接受新任务。
    p.join() # 阻塞等待所有已提交任务（含回调）执行完毕。

Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/opt/anaconda3/envs/gym3.10.10/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'add' on <module '__main__' (built-in)>


## wraps
- 任何时候你定义装饰器的时候，都应该使用 functools 库中的 @wraps 装饰器来注解底层包装函数

- wraps 装饰器能消除装饰器带来的副作用

### 装饰器的副作用


In [1]:
from functools import wraps

def decorator(func):
    # @wraps(func)
    def wrapper(*args, **kwargs):
        """wrapper func"""
        return func(*args, **kwargs)
    return wrapper

@decorator
def add(x, y):
    """add func"""
    return x + y


print(add.__name__)  # wrapper
print(add.__doc__)    # wrapper func
print(add)   # <function __main__.wrapper>
print(add(1,2))   

# 函数的 __name__ 属性变成了 wrapper ,实际上add 函数整个变成了 decorator(add)
# 这就是装饰器函数的副作用


wrapper
wrapper func
<function decorator.<locals>.wrapper at 0x10c7c00d0>
3


In [2]:
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """wrapper func"""
        return func(*args, **kwargs)
    return wrapper

@decorator
def add(x, y):
    """add func"""
    return x + y

print(add.__name__)  # add
print(add.__doc__)    # add func

print(add)   # <function __main__.add>
print(add(1,2))   


add
add func
<function add at 0x10cde0f70>
3


## lru缓存
- 当以相同的参数调用一个函数时，functools 模块中的这个装饰器将 n 次函数调用保存在缓存中，从而节省了大量时间。

In [3]:
import time

start = time.time()
def a_heavy_operation():
    time.sleep(3)
    return 11 + 22


print(a_heavy_operation())
print(a_heavy_operation())

print(time.time() - start)

33
33
6.007322788238525


In [4]:
import time
from functools import lru_cache

start = time.time()


@lru_cache()
def a_heavy_operation():
    time.sleep(3)
    return 11 + 22


print(a_heavy_operation())
print(a_heavy_operation())

print(time.time() - start)

33
33
3.0045361518859863


### 看看使用 lru 缓存如何使我们的代码运行得更快。
- Python 保存函数的缓存并检索缓存值，减少了我们的执行时间。

它彻底省掉了重复计算这一最昂贵的环节。核心原因就两条：
- 命中时直接拿“哈希表”结果，时间复杂度 O(1)
- 未命中时才真正跑原函数，且把结果写回缓存，以后永不重算

对于递归型爆炸任务（斐波那契、网格路径、最小编辑距离等）
- 时间从 O(kⁿ) 直接降到 O(n)
- 调用栈深度不变，但每层只算一次，后续全是哈希表瞬间返回

<b>lru_cache 更快，是因为它用 C 级哈希表+双向链表 把“曾经算过的结果”在 O(1) 时间内直接返回，从而消除了重复计算这一最耗时的步骤；对递归或高成本 IO 函数尤其明显。</b>

## singledispatch
- 要创建通用函数，可以使用 singledispatch。
- 通用函数是那些对各种数据类型执行相同操作的函数。

In [ ]:
def return_first_element(data):
    if isinstance(data, list):
        print(data[0])
    elif isinstance(data, str):
        print(data.split()[0])
    elif isinstance(data, dict):
        print(list(data.values())[0] )
    else:
        print(print(data))
        
return_first_element({"Age":20, "Height": 180})
return_first_element("Hello Mr Python")
return_first_element([12,432,563]) 

不建议在 Python 中使用 if/elif/else 语句来创建泛型函数。
- 那么，解决方案是什么？singledispatch

- 看看我们的代码现在干净多了；
- singledispatch 使得添加更多数据类型变得更容易，并且每个数据类型现在都有自己的位置，我们可以在其中对数据执行进一步的操作。

In [8]:
from functools import singledispatch

@singledispatch
def return_first_ele(data):
    return data


@return_first_ele.register(list)
def _(data):
    return data[0]


@return_first_ele.register(dict)
def _(data):
    return list(data.values())[0]


@return_first_ele.register(str)
def _(data):
    return data.split()[0]

return_first_ele({"Age":20, "Height": 180})
return_first_ele("Hello Mr Python")
return_first_ele([12,432,563]) 
return_first_el({12,31,1}) # 当没有数据类型与“set”匹配时，我们的函数如何充当回退函数。

{1, 12, 31}

## total_ordering
- total_ordering 装饰器在面向对象编程中节省了大量时间。

In [9]:
class Man:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __eq__(self, o):
        return self.age == o.age

    def __lt__(self, o):
        return self.age < o.age
    
    
obj = Man("Vivek", 20)
obj2 = Man("Alex", 24) 
obj = obj
obj == obj2

obj < obj2

obj >= obj2

TypeError: '>=' not supported between instances of 'Man' and 'Man'

### 我们的代码适用于 (==) 和 (<)，但当我们使用类中未定义的运算符时它不起作用。

- 鉴于我们至少创建了一个运算符 dunder 方法和 __eq__ 方法，@total_ordering 为我们的类生成了、>、=、>= 和更多比较运算符。

In [10]:
from functools import total_ordering

@total_ordering
class Man:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __eq__(self, o):
        return self.age == o.age

    def __lt__(self, o):
        return self.age < o.age
    
o = Man("Vivek", 20) 
b = Man("Alex", 24) 
o == b 
 
o >= b   
 
o <= b 

True